In [3]:
from sympy import symbols, Rational
from sympy.utilities.codegen import codegen
from sympy.codegen.rewriting import optimize, optims_c99
from sympy.simplify.cse_main import cse
import sympy as sp
from sympy import S
from sympy.printing.c import C99CodePrinter, Assignment
from sympy import init_printing
from codegen.codegen_utils import * 
init_printing()
printer = MyPrinter() 
from sympy.calculus.finite_diff import finite_diff_weights

In [4]:
def lagrange_1d(offsets, xi):
    """
    offsets: list of grid point positions relative to center (e.g. [-2,-1,0,1,2])
    xi: interpolation point (symbolic or numeric)
    returns list of L_j(xi)
    """
    s = offsets
    n = len(s)
    L = []
    for j in range(n):
        num = 1
        den = 1
        for m in range(n):
            if m != j:
                num *= (xi - s[m])
                den *= (s[j] - s[m])
        L.append(sp.simplify(num/den))
    return L

def lagrange_3d(offsets_x, offsets_y, offsets_z, xi, eta, zeta):
    """
    offsets_* : lists of offsets for each axis
    xi,eta,zeta : interpolation positions inside cell
    returns tensor-product weights W[i][j][k]
    """
    Lx = lagrange_1d(offsets_x, xi)
    Ly = lagrange_1d(offsets_y, eta)
    Lz = lagrange_1d(offsets_z, zeta)
    
    Wx = len(Lx)
    Wy = len(Ly)
    Wz = len(Lz)

    W = [[[sp.simplify(Lx[i]*Ly[j]*Lz[k]) 
           for k in range(Wz)]
           for j in range(Wy)]
           for i in range(Wx)]
    return W


In [ ]:
ORDER = 4 

cs = [S(-3),S(-1),S(1),S(3),S(5)]
#
l1 = [S(-5),S(-3),S(-1),S(1),S(3)]
l2 = [S(-7),S(-5),S(-3),S(-1),S(1)]
l3 = [S(-9),S(-7),S(-5),S(-3),S(-1)]

#
r1 = [S(-1),S(1),S(3),S(5),S(7)]
r2 = [S(1),S(3),S(5),S(7),S(9)]

off_map = {
    -3: l3, -2: l2, -1:l1, 0: cs, 1: r1, 2: r2
}
off_to_idx = {
    -3: 0, -2: 1, -1: 2, 0: 3, 1: 4, 2: 5
}

restrict_coeffs = {} 
for o,s in off_map.items():
    restrict_coeffs[o] = lagrange_1d(s,S(0))


s_prolong = [S(-7),S(-3),S(+1),S(+5)]
prolong_coeffs = lagrange_3d(s_prolong,s_prolong,s_prolong,S(0),S(0),S(0))


In [53]:
ORDER = 3 

cs = [S(-3),S(-1),S(1),S(3)]
#
l1 = [S(-5),S(-3),S(-1),S(1)]
l2 = [S(-7),S(-5),S(-3),S(-1)]

#
r1 = [S(-1),S(1),S(3),S(5)]
r2 = [S(1),S(3),S(5),S(7)]

off_map = {
    -2: l2, -1:l1, 0: cs, 1: r1, 2: r2
}
off_to_idx = {
    -2: 0, -1: 1, 0: 2, 1: 3, 2: 4
}

flat_restrict = []
restrict_coeffs = {} 
for o,s in off_map.items():
    restrict_coeffs[o] = lagrange_1d(s,S(0))
    for i in range(ORDER+1):
        flat_restrict.append(printer.doprint(restrict_coeffs[o][i]))

s_prolong_l = [S(-7),S(-3),S(+1),S(+5)]
s_prolong_u = [S(-5),S(-1),S(+3),S(+7)]
prolong_coeffs = {
    0: lagrange_1d(s_prolong_l,S(0)),
    1: lagrange_1d(s_prolong_u,S(0))
}
flat_prolong = [] 
for o in range(2):
    for i in range(ORDER+1):
            flat_prolong.append(printer.doprint(prolong_coeffs[o][i]))

In [45]:
ORDER = 2 

cs = [S(-3),S(-1),S(1)]

#
r1 = [S(-1),S(1),S(3)]
r2 = [S(1),S(3),S(5)]

off_map = {
    0: cs, 1: r1, 2: r2
}
off_to_idx = {
    0: 0, 1: 1, 2: 2
}

flat_restrict = []
restrict_coeffs = {} 
for o,s in off_map.items():
    restrict_coeffs[o] = lagrange_1d(s,S(0))
    for i in range(ORDER+1):
        flat_restrict.append(printer.doprint(restrict_coeffs[o][i]))


s_prolong_l = [S(-3),S(+1),S(+5)]
s_prolong_u = [S(-5),S(-1),S(+3)]

prolong_coeffs = {
    0: lagrange_1d(s_prolong_l,S(0)),
    1: lagrange_1d(s_prolong_u,S(0))
}
flat_prolong = [] 
for o in range(2):
    for i in range(ORDER+1):
            flat_prolong.append(printer.doprint(prolong_coeffs[o][i]))

In [50]:
# Join them with commas
values_str = ",\n    ".join(flat_restrict)

fill_rcoeffs_func = f'''
static void fill_fourth_order_restriction_coefficients(std::vector<double>& coeffs) {{
    coeffs.resize({len(values_str)});
    static const double raw_data[{len(values_str)}] = {{
        {values_str}
    }};
    coeffs.assign(raw_data, raw_data + {len(values_str)});
}}
'''

values_str = ",\n    ".join(flat_prolong)

fill_pcoeffs_func = f'''
static void fill_fourth_order_prolongation_coefficients(std::vector<double>& coeffs) {{
    coeffs.resize({len(values_str)});
    static const double raw_data[{len(values_str)}] = {{
        {values_str}
    }};
    coeffs.assign(raw_data, raw_data + {len(values_str)});
}}
'''

In [51]:
print(fill_rcoeffs_func)


static void fill_fourth_order_restriction_coefficients(std::vector<double>& coeffs) {
    coeffs.resize(168);
    static const double raw_data[168] = {
        1.0/16.0,
    -5.0/16.0,
    15.0/16.0,
    5.0/16.0,
    -1.0/16.0,
    9.0/16.0,
    9.0/16.0,
    -1.0/16.0,
    5.0/16.0,
    15.0/16.0,
    -5.0/16.0,
    1.0/16.0
    };
    coeffs.assign(raw_data, raw_data + 168);
}



In [52]:
print(fill_pcoeffs_func)


static void fill_fourth_order_prolongation_coefficients(std::vector<double>& coeffs) {
    coeffs.resize(124);
    static const double raw_data[124] = {
        -5.0/128.0,
    35.0/128.0,
    105.0/128.0,
    -7.0/128.0,
    -7.0/128.0,
    105.0/128.0,
    35.0/128.0,
    -5.0/128.0
    };
    coeffs.assign(raw_data, raw_data + 124);
}



In [198]:
# Create a flat list of all values
flat_prolong_values = []
for k in range(4):
    for j in range(4):
        for i in range(4):
            flat_prolong_values.append(prolong_coeffs[i][j][k])


import struct
# Ensure everything is a native python float
numeric_values = [float(v.evalf(17)) for v in flat_prolong_values]

with open("prolong_coeffs.bin", "wb") as f:
    # Use the length of the new numeric list
    f.write(struct.pack('d'*len(numeric_values), *numeric_values))

In [203]:
with open("test.bin", "wb") as f:
    f.write(struct.pack('<dd', 1.5,2.5))


In [55]:
import numpy as np 
xx = np.linspace(-2,2,17)
xc = (xx[1:] + xx[:-1])*0.5 
xx = np.linspace(-2,2,9)
xcC = (xx[1:] + xx[:-1])*0.5 
func = lambda x: 2*x**3 

def interpolate_function_1D(i_f,f_C):
    bi = i_f&1 
    i = i_f//2 
    ci  = bi-2 
    res = 0 
    for dd in range(4):
        
        res += prolong_coeffs[bi][dd] * f_C[i+dd+ci]
    return res 

print(interpolate_function_1D(6,func(xcC)))
print(func(xc[6]))


n = 16 

def find_stencil(pos):
    lb = pos 
    ub = n-1-pos 
    if lb<2:
        return 2 if lb==0 else 1
    if ub<2:
        return -2 if ub==0 else -1
    return 0 

def restrict_function_1D(i,FF):
    ox = find_stencil(i)
    cx = ox - 1
    
    res = 0 
    for dd in range(4):
        coeff = restrict_coeffs[ox][dd]
        res += coeff * FF[i+dd+cx]
    return res 
print(restrict_function_1D(6,func(xc)))
print(func(xcC[3]))

-0.105468750000000
-0.10546875
-0.0312500000000000
-0.03125


In [148]:
xx = np.linspace(-2,2,17)
yy = np.linspace(-2,2,17)
zz = np.linspace(-2,2,17)
xc = (xx[1:] + xx[:-1])*0.5 
yc = (yy[1:] + yy[:-1])*0.5 
zc = (zz[1:] + zz[:-1])*0.5 
X,Y,Z = np.meshgrid(xc,yc,zc,indexing="ij")
print(xc)
xx = np.linspace(-2,2,9)
yy = np.linspace(-2,2,9)
zz = np.linspace(-2,2,9)
xcC = (xx[1:] + xx[:-1])*0.5 
ycC = (yy[1:] + yy[:-1])*0.5 
zcC = (zz[1:] + zz[:-1])*0.5 
XC,YC,ZC = np.meshgrid(xcC,ycC,zcC,indexing="ij")
print(xcC)

[-1.875 -1.625 -1.375 -1.125 -0.875 -0.625 -0.375 -0.125  0.125  0.375
  0.625  0.875  1.125  1.375  1.625  1.875]
[-1.75 -1.25 -0.75 -0.25  0.25  0.75  1.25  1.75]


In [149]:
func = lambda x,y,z: x**3 + x*y*z*3.12 + 1 
FC = func(XC,YC,ZC)
FF = func(X,Y,Z)

In [169]:
i_f = 6
j_f = 6 
k_f = 6 

i_c = 3 
j_c = 3 
k_c = 3 

def interpolate_function(i_f,j_f,k_f,FC):
    bi = i_f&1 
    bj = j_f&1 
    bk = k_f&1

    i = i_f//2 
    j = j_f//2 
    k = k_f//2  

    ci = bi-2 
    cj = bj-2
    ck = bk-2
    
    res = 0 
    for dd in range(64):
        di = dd&3 
        dj = (dd>>2)&3
        dk = dd>>4 

        r_di = di if bi==0 else 3-di 
        r_dj = dj if bj==0 else 3-dj 
        r_dk = dk if bk==0 else 3-dk 

        res += prolong_coeffs[r_di][r_dj][r_dk] * FC[i+di+ci,j+dj+cj,k+dk+ck]
    return res 

n = 16 

def find_stencil(pos):
    lb = pos 
    ub = n-1-pos 
    if lb<2:
        return 2 if lb==0 else 1
    if ub<2:
        return -2 if ub==0 else -1
    return 0 

def restrict_function(i,j,k,FF):
    ox = find_stencil(i)
    oy = find_stencil(j)
    oz = find_stencil(k)

    cx = ox - 1
    cy = oy - 1
    cz = oz - 1
    
    res = 0 
    for dd in range(64):
        di = dd&3 
        dj = (dd>>2)&3
        dk = dd>>4 
        coeff = coeffs[(ox,oy,oz)][di][dj][dk]
        res += coeff * FF[i+di+cx,j+dj+cy,k+dk+cz]
    return res 

In [170]:
restrict_function(i_f,j_f,k_f,FF)

In [171]:
print(FC[i_c,j_c,k_c])

0.935625


In [154]:
interpolate_function(i_f,j_f,k_f,FC)

In [155]:
FF[i_f,j_f,k_f]